# CineMatch API Server

This notebook runs the FastAPI backend that serves the Next.js frontend.
It reuses all engine logic from `gradio_app.ipynb` cells and exposes clean REST endpoints.

**Setup:** Run cells 1-3 first (dependencies, engine import, API definition), then cell 4 to start the server.

In [ ]:
# Cell 1: Dependencies
!pip install -q fastapi uvicorn pycloudflared pymongo

In [ ]:
# Cell 2: Import the engine
# This cell must be run AFTER all the gradio_app.ipynb cells that define
# the engine functions (mongo, embeddings, recommendations, etc.)
#

import json
import uuid
from datetime import datetime, timezone
from typing import Optional
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel


In [ ]:
# Cell 3: Define FastAPI app and routes

app = FastAPI(title="CineMatch API", version="1.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # For development; restrict in production
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ─── In-memory session store (keyed by session_id) ─────────
# In production this would be Redis; for demo, dict is fine.
SESSIONS: dict[str, dict] = {}


def _get_session(session_id: str) -> dict:
    if session_id not in SESSIONS:
        raise HTTPException(status_code=404, detail="Session not found")
    return SESSIONS[session_id]


def _session_to_response(session: dict) -> dict:
    """Convert internal session to the UserSession shape the frontend expects."""
    feedback = session.get('onboarding_feedback', {})
    slate = session.get('slate', [])
    like_count = sum(1 for v in feedback.values() if str(v) == 'like')
    total = len(slate)
    rated = len(feedback)
    is_complete = rated >= total and total > 0
    is_ready = is_complete and like_count >= GRADIO_MIN_ONBOARDING_LIKES
    return {
        'session_id': session.get('session_id', ''),
        'user_id': session.get('user_id', ''),
        'identifier': session.get('identifier', ''),
        'is_returning': session.get('_is_returning', False),
        'profile': session.get('profile', {}),
        'onboarding_complete': is_complete,
        'onboarding_index': int(session.get('onboarding_index', 0)),
        'onboarding_total': total,
        'onboarding_likes': like_count,
        'min_likes_needed': GRADIO_MIN_ONBOARDING_LIKES,
        'has_recommendations': bool(session.get('recommendation_pool')),
    }


def _movie_from_record(record: dict) -> dict:
    """Convert a slate/catalog record to the Movie shape."""
    genres_raw = record.get('genres', record.get('genre_names', ''))
    if isinstance(genres_raw, str):
        genres = [g.strip() for g in genres_raw.split(',') if g.strip()]
    elif isinstance(genres_raw, list):
        genres = genres_raw
    else:
        genres = []
    return {
        'id': int(record.get('id', 0)),
        'title': record.get('title', ''),
        'original_title': record.get('original_title', ''),
        'year': record.get('year', record.get('release_year')),
        'poster_path': record.get('poster_path', ''),
        'backdrop_path': record.get('backdrop_path', ''),
        'overview': record.get('overview', ''),
        'original_language': record.get('original_language', ''),
        'genres': genres,
        'primary_genre': record.get('primary_genre', genres[0] if genres else ''),
        'vote_average': record.get('vote_average'),
        'vote_count': record.get('vote_count'),
        'director': record.get('director', ''),
        'imdb_rating': record.get('imdb_rating'),
        'runtime': record.get('runtime'),
    }


# ─── Request/Response Models ────────────────────────────────

class LoginRequest(BaseModel):
    email: str

class SlateRequest(BaseModel):
    session_id: str
    languages: list[str] = ['en']
    genres: list[str] = []
    age_group: str = '18-24'
    region: str = 'US'
    include_classics: bool = True
    semantic_index: str = 'tmdb_bge_m3'

class RateRequest(BaseModel):
    session_id: str
    tmdb_id: int
    rating: str  # 'like', 'okay', 'dislike', 'not_watched'

class NavRequest(BaseModel):
    session_id: str
    direction: str  # 'prev' or 'next'

class RecommendationRequest(BaseModel):
    session_id: str
    languages: list[str] = ['en']
    genres: list[str] = []
    semantic_index: str = 'tmdb_bge_m3'

class ActionRequest(BaseModel):
    session_id: str
    tmdb_id: int
    action: str  # 'like', 'okay', 'dislike'


# ─── Routes ─────────────────────────────────────────────────

@app.post('/api/login')
async def login(req: LoginRequest):
    email = req.email.strip().lower()
    if not email:
        raise HTTPException(400, 'Email is required.')
    session = gradio_empty_session(identifier=email)
    SESSIONS[session['session_id']] = session
    return _session_to_response(session)


@app.post('/api/onboarding/slate')
async def build_slate(req: SlateRequest):
    session = _get_session(req.session_id)
    # Call the engine's slate builder
    outputs = gradio_build_slate(
        session,
        req.age_group,
        req.region,
        req.languages,
        req.genres,
        req.include_classics,
        req.semantic_index,
    )
    # outputs[0] is the updated session
    session = outputs[0]
    SESSIONS[req.session_id] = session
    return _onboarding_state(session)


@app.post('/api/onboarding/rate')
async def rate_onboarding(req: RateRequest):
    session = _get_session(req.session_id)
    # Store the rating
    session['onboarding_feedback'][str(req.tmdb_id)] = req.rating
    # Log interaction
    mongo_log_interaction(
        user_id=session.get('user_id', 'anonymous'),
        tmdb_id=req.tmdb_id,
        action=req.rating,
        context='onboarding',
        metadata={'semantic_index': session.get('semantic_index_name', '')},
    )
    # Auto-advance to next unrated
    slate = session.get('slate', [])
    current = int(session.get('onboarding_index', 0))
    if current < len(slate) - 1:
        session['onboarding_index'] = current + 1
    _mongo_sync_session(session)
    SESSIONS[req.session_id] = session
    return _onboarding_state(session)


@app.post('/api/onboarding/nav')
async def nav_onboarding(req: NavRequest):
    session = _get_session(req.session_id)
    slate = session.get('slate', [])
    current = int(session.get('onboarding_index', 0))
    if req.direction == 'prev' and current > 0:
        session['onboarding_index'] = current - 1
    elif req.direction == 'next' and current < len(slate) - 1:
        session['onboarding_index'] = current + 1
    SESSIONS[req.session_id] = session
    return _onboarding_state(session)


@app.post('/api/recommendations')
async def generate_recommendations(req: RecommendationRequest):
    session = _get_session(req.session_id)
    # Update profile preferences
    profile = dict(session.get('profile', {}))
    profile['preferred_languages'] = req.languages
    profile['preferred_genres'] = req.genres
    session['profile'] = profile
    session['semantic_index_name'] = resolve_semantic_index_name(req.semantic_index)
    # Call the engine
    outputs = gradio_generate_recommendations(
        session,
        profile.get('age_group', '18-24'),
        profile.get('region', 'US'),
        profile.get('preferred_languages', ['en']),
        profile.get('preferred_genres', []),
        profile.get('include_classics', True),
        req.semantic_index
    )
    session = outputs[0]
    SESSIONS[req.session_id] = session
    return _recommendation_page(session)


@app.post('/api/recommendations/action')
async def recommendation_action(req: ActionRequest):
    session = _get_session(req.session_id)
    session['recommendation_feedback'][str(req.tmdb_id)] = req.action
    session['actions_since_refresh'] = session.get('actions_since_refresh', 0) + 1
    if req.action == 'dislike':
        session['negative_actions_since_refresh'] = session.get('negative_actions_since_refresh', 0) + 1
    if req.action in ('like', 'okay'):
        session['positive_actions_since_refresh'] = session.get('positive_actions_since_refresh', 0) + 1
    mongo_log_interaction(
        user_id=session.get('user_id', 'anonymous'),
        tmdb_id=req.tmdb_id,
        action=req.action,
        context='recommendation',
        metadata={'semantic_index': session.get('semantic_index_name', '')},
    )
    _mongo_sync_session(session)
    SESSIONS[req.session_id] = session
    return _recommendation_page(session)


@app.get('/api/history')
async def get_history(session_id: str):
    session = _get_session(session_id)
    items = []
    # Onboarding history
    for tmdb_id_str, rating in session.get('onboarding_feedback', {}).items():
        tmdb_id = int(tmdb_id_str)
        title = GRADIO_TITLE_BY_ID.get(tmdb_id, f'Movie {tmdb_id}')
        poster = GRADIO_POSTER_BY_ID.get(tmdb_id, '')
        items.append({
            'tmdb_id': tmdb_id,
            'title': title,
            'poster_path': poster,
            'rating': str(rating),
            'context': 'onboarding',
        })
    # Recommendation history
    for tmdb_id_str, rating in session.get('recommendation_feedback', {}).items():
        tmdb_id = int(tmdb_id_str)
        title = GRADIO_TITLE_BY_ID.get(tmdb_id, f'Movie {tmdb_id}')
        poster = GRADIO_POSTER_BY_ID.get(tmdb_id, '')
        items.append({
            'tmdb_id': tmdb_id,
            'title': title,
            'poster_path': poster,
            'rating': str(rating),
            'context': 'recommendation',
        })
    return items


# ─── Helpers ────────────────────────────────────────────────

def _onboarding_state(session: dict) -> dict:
    slate = session.get('slate', [])
    feedback = session.get('onboarding_feedback', {})
    current_index = int(session.get('onboarding_index', 0))
    like_count = sum(1 for v in feedback.values() if str(v) == 'like')
    total = len(slate)
    rated = len(feedback)
    is_complete = rated >= total and total > 0
    is_ready = is_complete and like_count >= GRADIO_MIN_ONBOARDING_LIKES
    movie = None
    if slate and 0 <= current_index < total:
        movie = _movie_from_record(slate[current_index])
    counts = {}
    for v in feedback.values():
        v_str = str(v)
        counts[v_str] = counts.get(v_str, 0) + 1
    return {
        'session': _session_to_response(session),
        'movie': movie,
        'feedback_counts': counts,
        'is_complete': is_complete,
        'is_ready': is_ready,
    }


def _recommendation_page(session: dict) -> dict:
    pool = session.get('recommendation_pool', [])
    feedback = session.get('recommendation_feedback', {})
    # Filter out already-actioned movies
    visible = [r for r in pool if str(r.get('id', r.get('tmdb_id', ''))) not in feedback]
    # Take top N for the page
    page_size = 20
    page = visible[:page_size]
    movies = [_movie_from_record(r) for r in page]
    return {
        'session': _session_to_response(session),
        'movies': movies,
        'status': f'{len(visible)} recommendations remaining.',
        'total_pool_size': len(visible),
    }


# Check if GRADIO_POSTER_BY_ID exists, create placeholder if not
try:
    GRADIO_POSTER_BY_ID
except NameError:
    GRADIO_POSTER_BY_ID = {}

print('FastAPI app defined with', len(app.routes), 'routes.')

In [ ]:
# Cell 4: Start the server with Cloudflare Tunnel
import threading
import uvicorn
from pycloudflared import try_cloudflare

PORT = 8000

# Start Cloudflare tunnel
tunnel = try_cloudflare(port=PORT)
print(f'\n──────────────────────────────────────────')
print(f'  CineMatch API is live!')
print(f'  Public URL: {tunnel}')
print(f'  Use this as NEXT_PUBLIC_API_URL in your .env.local')
print(f'──────────────────────────────────────────\n')

# Run uvicorn in a thread so the notebook doesn't block
config = uvicorn.Config(app, host='0.0.0.0', port=PORT, log_level='info')
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
print('Server running in background. Notebook is still interactive.')